# 04 — Inlet Bellmouth Design

**Purpose:** Design the complete annular inlet assembly consisting of (i) the ISO 5801 outer bellmouth lip, (ii) the semi-ellipsoidal hub centerbody / spinner, and (iii) the upstream settling section with honeycomb and turbulence screen. Evaluate the total-pressure loss budget, turbulence intensity at the rotor face, annular boundary-layer blockage, and locate the ISO 5801 mass-flow measurement plane.

**Inputs:** IGV geometry from notebook 03 (`igv_geometry()` output).

**Outputs:**
- Meridional contours of both stream surfaces
- Loss budget breakdown
- Axial station map (all upstream components referenced to rotor LE)
- Total inlet assembly length

**References:** ISO 5801:2017; Mehta & Bradshaw (1979) Aeronautical J.; Bell & Mehta (1988) NASA CR-177488; Hoerner (1965) Fluid Dynamic Drag.

In [ ]:
import sys, pathlib

# Locate repo root (directory that contains src/) regardless of
# where the notebook file sits (repo root or notebooks/ subfolder)
_here = pathlib.Path.cwd()
_root = next(
    (p for p in [_here, *_here.parents] if (p / "src").is_dir()),
    _here,
)
sys.path.insert(0, str(_root))
pathlib.Path(_root / "figures").mkdir(exist_ok=True)


import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from src.igv       import igv_geometry, meanline_with_igv
from src.bellmouth import bellmouth_design, print_bellmouth_summary

plt.rcParams.update({'figure.dpi': 130, 'font.size': 10})
print('Imports OK')

In [ ]:
# ── Load design point from notebook 01 ─────────────────────
import json as _json, pathlib as _pl

_dp_path = _pl.Path(_root / "design_point.json")
if not _dp_path.exists():
    raise FileNotFoundError(
        "design_point.json not found — run notebook 01 first.")

dp = _json.loads(_dp_path.read_text())

# Unpack for convenience
D_TIP = dp["D_tip"]
N_RPM = dp["N_RPM"]
PR    = dp["PR"]
NU    = dp["nu"]
PHI   = dp["phi"]
ETA   = dp["eta_is"]
print(f"Design point: D_tip={D_TIP*1000:.0f} mm  N={N_RPM} RPM  PR={PR}  nu={NU}  phi={PHI}  eta={ETA}")


## 1. Build from IGV design point

In [ ]:
igv   = igv_geometry(D_tip=D_TIP, nu=NU, N_RPM=N_RPM, phi=PHI, alpha1_deg=0.0)
rotor = meanline_with_igv(igv, PR=PR, eta_is=ETA)

bell  = bellmouth_design(
    igv,
    contraction_ratio   = 4.0,
    Cd                  = 0.99,
    n_screens           = 1,
    screen_wire_d       = 0.5e-3,
    screen_mesh         = 16.0,
    honeycomb_cell      = 6.35e-3,
    honeycomb_L_D       = 8.0,
    centerbody_fineness = 1.2,
)

print_bellmouth_summary(bell, rotor_dP0_Pa=rotor['dP0_rotor_Pa'])

## 2. Meridional contour — both stream surfaces

In [ ]:
outer = bell['outer']
cb    = bell['centerbody']

# IGV LE is the origin of the bellmouth coordinate system (x=0 here)
# In the bellmouth module, x_bell_exit = x_igv_LE (< 0 in shaft coords)
# We plot in the local frame where x=0 is the bellmouth exit / IGV LE.
x_bell_exit  = 0.0
x_bell_entry = -outer['L_total_m']

x_outer = outer['x_contour'] + x_bell_entry
r_outer = outer['r_contour']

# Centerbody in local frame (nose tip upstream, shoulder at x=0)
x_cb = -(cb['L_nose_m'] - cb['x_contour'])  # shift so shoulder = 0
r_cb =   cb['r_contour']

fig, ax = plt.subplots(figsize=(12, 5))

# Outer wall (positive and negative r)
ax.plot(x_outer * 1000, r_outer * 1000,  color='#185FA5', lw=2.5, label='Outer bellmouth')
ax.plot(x_outer * 1000, -r_outer * 1000, color='#185FA5', lw=2.5)

# Centerbody
ax.plot(x_cb * 1000, r_cb * 1000,  color='#D85A30', lw=2.5, label='Hub centerbody')
ax.plot(x_cb * 1000, -r_cb * 1000, color='#D85A30', lw=2.5)

# Fill annulus
ax.fill_between(x_cb * 1000, r_cb * 1000, r_outer * 1000,
                alpha=0.08, color='#185FA5', label='Annular passage')
ax.fill_between(x_cb * 1000, -r_cb * 1000, -r_outer * 1000, alpha=0.08, color='#185FA5')

# Station markers
x_meas_loc = (bell['x_meas_mm'] - bell['x_bell_exit_mm'])
ax.axvline(x_meas_loc, color='green', ls='--', lw=1.5, label='ISO 5801 meas. plane')
ax.axvline(0, color='gray', ls=':', lw=1.5, label='Bellmouth exit / IGV LE')

ax.set(xlabel='Axial position [mm]', ylabel='Radius [mm]',
       title='Annular Bellmouth Inlet — Meridional Contour')
ax.legend(fontsize=9, loc='upper left')
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig(str(_root / 'figures') + '/04_bellmouth_contour.png', dpi=130, bbox_inches='tight')
plt.show()

## 3. Loss budget bar chart

In [ ]:
sources = ['Outer bellmouth\nfriction', 'Centerbody\nnose drag',
           'Screen ×1', 'Honeycomb']
values  = [bell['dP0_bell_Pa'], bell['dP0_centerbody_Pa'],
           bell['dP0_screen_Pa'], bell['dP0_honey_Pa']]
colors  = ['#185FA5', '#D85A30', '#1D9E75', '#533AB7']

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(sources, values, color=colors, edgecolor='white', linewidth=0.8)
ax.bar_label(bars, fmt='%.1f Pa', padding=3, fontsize=9)
ax.set(ylabel='Total pressure loss ΔP₀ [Pa]',
       title=f'Inlet Loss Budget  (Total = {bell["dP0_total_Pa"]:.1f} Pa)')
ax.grid(True, axis='y', alpha=0.35)
plt.tight_layout()
plt.savefig(str(_root / 'figures') + '/04_loss_budget.png', dpi=130, bbox_inches='tight')
plt.show()

print(f'Total inlet ΔP₀       : {bell["dP0_total_Pa"]:.1f} Pa')
print(f'As % of rotor ΔP₀     : {bell["dP0_total_Pa"]/rotor["dP0_rotor_Pa"]*100:.2f}%')
print(f'As % of dynamic head  : {bell["zeta_total"]*100:.3f}%')

## 4. Settling section sensitivity

Show how turbulence intensity at the rotor face varies with contraction ratio.

In [ ]:
CR_range = np.linspace(2, 8, 30)
Tu_range = []
dP_range = []

for cr in CR_range:
    b = bellmouth_design(igv, contraction_ratio=cr)
    Tu_range.append(b['Tu_throat'] * 100)
    dP_range.append(b['dP0_total_Pa'])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle('Settling Section — Contraction Ratio Sensitivity', fontweight='bold')

axes[0].plot(CR_range, Tu_range, '#1D9E75', lw=2)
axes[0].axvline(4.0, color='gray', ls=':', label='CR = 4.0 chosen')
axes[0].set(xlabel='Contraction ratio CR', ylabel='Tu at rotor face [%]')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.35)

axes[1].plot(CR_range, dP_range, '#D85A30', lw=2)
axes[1].axvline(4.0, color='gray', ls=':')
axes[1].set(xlabel='Contraction ratio CR', ylabel='Total inlet ΔP₀ [Pa]')
axes[1].grid(True, alpha=0.35)

plt.tight_layout()
plt.savefig(str(_root / 'figures') + '/04_settling_sensitivity.png', dpi=130, bbox_inches='tight')
plt.show()

## 5. Full axial station map

In [ ]:
stations_map = [
    ('Settling section entry',      bell['x_settle_entry_mm']),
    ('Centerbody nose tip',         bell['x_cb_nose_tip_mm']),
    ('Bellmouth lip (outer entry)', bell['x_bell_entry_mm']),
    ('ISO 5801 meas. plane',        bell['x_meas_mm']),
    ('Bellmouth exit / IGV LE',     bell['x_bell_exit_mm']),
    ('IGV trailing edge',           bell['x_igv_TE_mm']),
    ('Rotor leading edge',          bell['x_rotor_LE_mm']),
]

print('Axial station map  (x = 0 at rotor LE, upstream = negative)')
print('─' * 58)
for name, x in sorted(stations_map, key=lambda s: s[1]):
    print(f'  {name:<38}: x = {x:+8.1f} mm')
print(f'\nTotal inlet assembly length : {bell["L_inlet_total_mm"]:.0f} mm')

---
**Proceed to** `05_shaft_bearings_campbell.ipynb`.